# Vector Search Basics

In [0]:
%pip install -q -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import os 
import yaml
from dbruntime.databricks_repl_context import get_context

#for development purposes only
os.environ["DATABRICKS_TOKEN"] = get_context().apiToken
os.environ["DATABRICKS_HOST"] = "https://" + get_context().browserHostName

with open('./vs_intro.yaml', 'r') as file:
    config = yaml.safe_load(file)

databricks_config = config["databricks_config"]
catalog = databricks_config["catalog"]
schema = databricks_config["schema"]

vs_config = config["retriever_config"]

doc_table_name = vs_config["document_table_name"]
vs_endpoint_name = vs_config["vector_search_endpoint"]
vs_index_name = vs_config["vector_search_index"]
embedding_model = vs_config["embedding_model"]

llm_endpoint = config["agent_config"]["endpoint_name"]

In [0]:
from databricks.vector_search.client import VectorSearchClient

#optional params to instantiate with specific auth creds
vsc = VectorSearchClient(
  # disable_notice=True,
  # workspace_url = "https://e2-demo-field-eng.cloud.databricks.com/",
  # service_principal_client_id = dbutils.secrets.get("felix-flory", "SERVICE_PRINCIPAL_ID"),
  # service_principal_client_secret = dbutils.secrets.get("felix-flory", "SERVICE_PRINCIPAL_SECRET"),
  )
index = vsc.get_index(vs_endpoint_name, f"{catalog}.{schema}.{vs_index_name}")

In [0]:
display((spark.table(f"`{catalog}`.`{schema}`.`{doc_table_name}`")))

In [0]:
columns = spark.table(f"`{catalog}`.`{schema}`.`{doc_table_name}`").columns
filters = {
    "company": "american express",
}

In [0]:
results = index.similarity_search(
  query_text="What was the last earnings results from American Express",
  columns=columns,
  num_results=10,
  filters = filters,
  query_type = "HYBRID", # "ANN" or HYBRID"
  score_threshold = 0.50
)

if results["result"]["row_count"] >0:
  display(results["result"]["data_array"])
else:
  print("No records")


# Integrate with Langchain


In [0]:
import mlflow
from databricks_langchain import VectorSearchRetrieverTool, ChatDatabricks

mlflow.langchain.autolog()

# Initialize the retriever tool.
vs_tool = VectorSearchRetrieverTool(
  index_name=f"{catalog}.{schema}.{vs_index_name}",
  tool_name="docs_retriever",
  tool_description="Retrieves information about SEC filings",
)

# Run a query against the vector search index locally for testing
vs_tool.invoke("What was the last earnings results from American Express", disable_notice=True)

In [0]:
# Bind the retriever tool to your Langchain LLM of choice
llm = ChatDatabricks(endpoint=llm_endpoint, temperature = 0.0)
llm_with_tools = llm.bind_tools([vs_tool])

# Chat with your LLM to test the tool calling functionality
llm_with_tools.invoke("What were the key risks highlighted for American Express in their 2022 last earnings?")

# Create a ReACT Agent

In [0]:
from langgraph.prebuilt import create_react_agent

# Initialize the retriever tool.
vs_tool = VectorSearchRetrieverTool(
  index_name=f"{catalog}.{schema}.{vs_index_name}",
  tool_name="docs_retriever",
  tool_description="Retrieves information about SEC filings",
)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

doc_retrieval_agent = create_react_agent(
    model=llm,
    tools=[vs_tool, multiply],
    prompt="You are a document agent that has access to a retriever tool to look up financial documents. Use the documents available to CONCISELY ANSWER the provided question. Be as efficient as possible in your approach and minimize retriever calls where possible",
    name="document_agent",
)

In [0]:
response = doc_retrieval_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What were the key risks highlighted for American Express in their 2022 last earnings?",
            }
        ]
    }
)

In [0]:
print(response["messages"][-1].content)